# XML-based BT generation

In [ ]:
import openai
import json
import os
from pathlib import Path

from openai import OpenAI

# Resolve paths from either the repository root or the dataset/ folder.
repo_root = Path.cwd()
if not (repo_root / "keys" / "keys.json").exists() and repo_root.name == "dataset":
    repo_root = repo_root.parent

keys_path = repo_root / "keys" / "keys.json"
with keys_path.open("r", encoding="utf-8") as f:
    keys = json.load(f)

client = OpenAI(api_key=keys["OPENAI_API_KEY"])
DATASET_DIR = repo_root / "dataset"

In [ ]:
import glob
import os

# Source XML behavior trees from the TSE dataset. Replace this path with the local TSE checkout.
dir_path = "PATH_TO_TSE_DATASET"
filelist = sorted(glob.glob(os.path.join(dir_path, "**/*.xml"), recursive=True))
print(f"Found {len(filelist)} XML files.")

In [ ]:
from tqdm import tqdm

# Verify that every XML file can be decoded before sending it to the generation model.
unreadable_files = []
for path in tqdm(filelist):
    try:
        with open(path, "r", encoding="utf-8") as file:
            _ = file.read()
    except UnicodeDecodeError:
        unreadable_files.append(path)

print(f"Unreadable XML files: {len(unreadable_files)}")
for path in unreadable_files:
    print(path)

In [ ]:
instruction = """
You are a helpful assistant that can help me with my tasks.
Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.\n\n
"""

example_user_prompt = """
You are given a behavior tree in XML format and your task is to generate three alternative versions of the tree. 
Each tree must:
- Use the same actions as the provided tree below.
- The tree must be compatible with the BehaviorTree.CPP library.
- Be creative and reflect diverse robotic tasks like patrolling, exploring, or prioritizing goals.
- Include all node types (Decorators, FallBacks, Sequences, and Parallel) used differently in each tree but none necessarily in the same way as the original tree.
- Ensure a unique combination of hierarchical depth, branching factor, and node arrangements.
- The first tag of the tree is <root BTCPP_format="4"> and it is mandatory. It should contain 1 or more tags <BehaviorTree>.
- The tag <BehaviorTree> should have the attribute [ID].
- The tag <root> should contain the attribute [main_tree_to_execute]. The attribute [main_tree_to_execute] is mandatory if the file contains multiple <BehaviorTree>, optional otherwise.
- After the attribute [main_tree_to_execute] there is always the attribute BTCPP_format="4".
- ControlNodes contain 1 to N children, DecoratorNodes and Subtrees contain only 1 child, ActionNodes and ConditionNodes have no child.
- Avoid repetitive task sequences and vary node parameters.
- Be sure to generate behavior trees syntactically correct, compatible with the BehaviorTree.CPP library.
- Make sure each tree is tailored to realistic robotic scenarios.


<root BTCPP_format="4">
    <BehaviorTree ID="main">
        <Sequence> 
            <ForceSuccess>
                <Parallel success_count="2" failure_count="1">  
                    <isGoalReachable prob="1.0"/>                
                    <MoveTo name="go_to_station_A" location="Station A"/> 
                </Parallel>
            </ForceSuccess>
            <ForceSuccess>
                <Parallel success_count="2" failure_count="1">
                    <isGoalReachable prob="0.3"/>    
                    <MoveTo name="go_to_station_B" location="Station B"/>    
                </Parallel>
            </ForceSuccess>
            <MoveTo name="go_to_parking"   location="Parking"/>          
        </Sequence>
    </BehaviorTree>
</root>
"""

example_assistant_output= """
<root BTCPP_format="4">
    <BehaviorTree ID="main">
        <Fallback> 
            <Sequence> 
                <isGoalReachable prob="0.75"/>  
                <MoveTo name="move_to_restaurant" location="Restaurant"/>  
            </Sequence>
             <Sequence> 
                <isGoalReachable prob="0.5"/>  
                <MoveTo name="move_to_bar" location="Bar"/>  
            </Sequence>
            <Sequence> 
                <isGoalReachable prob="0.25"/>  
                <MoveTo name="move_to_bench" location="Bench"/>  
            </Sequence>   
        </Fallback>
    </BehaviorTree>
</root>


<root BTCPP_format="4">
    <BehaviorTree ID="main">
        <Sequence> 
            <MoveTo name="navigate_to_room1" location="Room1"/>  
            <MoveTo name="navigate_to_room2" location="Room2"/> 
            <isGoalReachable prob="0.5"/> 
            <MoveTo name="move_to_workstation" location="Workstation"/>
        </Sequence>
    </BehaviorTree>
</root>


<root BTCPP_format="4">
    <BehaviorTree ID="main">
        <Sequence>
            <RepeatUntilFailure>
                <Parallel success_count="2" failure_count="1">
                    <isGoalReachable prob="0.9"/>
                    <MoveTo name="go_to_station_C" location="Station C"/>
                </Parallel>
            </RepeatUntilFailure>
            <Fallback>
                <SucceedsIf>
                    <Parallel success_count="1" failure_count="2">
                        <isGoalReachable prob="0.4"/>
                        <MoveTo name="go_to_station_D" location="Station D"/>
                    </Parallel>
                </SucceedsIf>
                <MoveTo name="go_to_station_A" location="Station A"/>
            </Fallback>
            <MoveTo name="go_to_maintenance" location="Maintenance Area"/>
        </Sequence>
    </BehaviorTree>
</root>
"""

In [ ]:
user_prompt = """
You are given a behavior tree in XML format and your task is to generate three alternative versions of the tree. 
Each tree must:
- Use the same actions as the provided tree below.
- The tree must be compatible with the BehaviorTree.CPP library.
- Be creative and reflect diverse robotic tasks like patrolling, exploring, or prioritizing goals.
- Include all node types (Decorators, FallBacks, Sequences, and Parallel) used differently in each tree but none necessarily in the same way as the original tree.
- Ensure a unique combination of hierarchical depth, branching factor, and node arrangements.
- The first tag of the tree is <root BTCPP_format="4"> and it is mandatory. It should contain 1 or more tags <BehaviorTree>.
- The tag <BehaviorTree> should have the attribute [ID].
- The tag <root> should contain the attribute [main_tree_to_execute]. The attribute [main_tree_to_execute] is mandatory if the file contains multiple <BehaviorTree>, optional otherwise.
- After the attribute [main_tree_to_execute] there is always the attribute BTCPP_format="4".
- ControlNodes contain 1 to N children, DecoratorNodes and Subtrees contain only 1 child, ActionNodes and ConditionNodes have no child.
- Avoid repetitive task sequences and vary node parameters.
- Be sure to generate behavior trees syntactically correct, compatible with the BehaviorTree.CPP library.
- Make sure each tree is tailored to realistic robotic scenarios.
"""

## First run

In [ ]:
from pydantic import BaseModel
from tqdm import tqdm


class BT(BaseModel):
    behavior_tree: str


class BT_list(BaseModel):
    bt: list[BT]


json_list = []
next_id = 0
original_id_bt = 0
total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

# First augmentation pass: generate three alternative BTs for each source XML tree.
for path in tqdm(filelist):
    with open(path, "r", encoding="utf-8") as file:
        xml_tree = file.read()
    prompt = user_prompt + "\n\n" + xml_tree

    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": example_user_prompt},
                {"role": "assistant", "content": example_assistant_output},
                {"role": "user", "content": prompt}
            ],
            top_p=0.99,  # Encourage varied BT structures while keeping deterministic schema parsing.
            response_format=BT_list
        )

        output = completion.choices[0].message.parsed

        for bt_item in output.bt:
            json_list.append({
                "id": next_id,
                "original_bt_id": original_id_bt,
                "behavior_tree": bt_item.behavior_tree
            })
            next_id += 1

        usage = completion.usage
        total_usage["prompt_tokens"] += usage.prompt_tokens
        total_usage["completion_tokens"] += usage.completion_tokens
        total_usage["total_tokens"] += usage.total_tokens

    except openai.OpenAIError as e:
        print(f"OpenAIError on BT #{original_id_bt} ({path}): {e}")

    except Exception as e:
        print(f"Unexpected error on BT #{original_id_bt} ({path}): {e}")

    original_id_bt += 1

print("Total Token Usage:")
print(json.dumps(total_usage, indent=4))

In [ ]:
# Save the first pass with the same filename consumed by the second pass below.
file_name = DATASET_DIR / "synthetic_bt_dataset_first_run.json"
with file_name.open("w", encoding="utf-8") as json_file:
    json.dump(json_list, json_file, indent=4, ensure_ascii=False)

## Second run

In [ ]:
# Load first-pass outputs so each generated BT can be augmented once more.
dataset_path = DATASET_DIR / "synthetic_bt_dataset_first_run.json"
with dataset_path.open("r", encoding="utf-8") as json_file:
    data = json.load(json_file)

In [ ]:
from pydantic import BaseModel
from tqdm import tqdm


class BT(BaseModel):
    behavior_tree: str


class BT_list(BaseModel):
    bt: list[BT]


json_list = []
next_id = 0
total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

# Second augmentation pass: expand each first-pass BT while preserving the original source ID.
for item in tqdm(data):
    bt_id = item["original_bt_id"]
    bt = item["behavior_tree"]
    prompt = user_prompt + "\n\n" + bt

    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": example_user_prompt},
                {"role": "assistant", "content": example_assistant_output},
                {"role": "user", "content": prompt}
            ],
            top_p=0.99,  # Encourage diverse alternatives while enforcing BT_list parsing.
            response_format=BT_list
        )

        output = completion.choices[0].message.parsed

        for bt_item in output.bt:
            json_list.append({
                "id": next_id,
                "original_bt_id": bt_id,
                "behavior_tree": bt_item.behavior_tree
            })
            next_id += 1

        usage = completion.usage
        total_usage["prompt_tokens"] += usage.prompt_tokens
        total_usage["completion_tokens"] += usage.completion_tokens
        total_usage["total_tokens"] += usage.total_tokens

    except openai.OpenAIError as e:
        print(f"OpenAIError on generated BT from original #{bt_id}: {e}")
        continue

    except Exception as e:
        print(f"Unexpected error on generated BT from original #{bt_id}: {e}")
        continue

print("Total Token Usage:")
print(json.dumps(total_usage, indent=4))

In [ ]:
file_name = DATASET_DIR / "synthetic_bt_dataset_second_run.json"
with file_name.open("w", encoding="utf-8") as json_file:
    json.dump(json_list, json_file, indent=4, ensure_ascii=False)

In [ ]:
# Optional manual spot-check of generated XML strings.
file_name = DATASET_DIR / "synthetic_bt_dataset_second_run.json"
with file_name.open("r", encoding="utf-8") as json_file:
    json_list = json.load(json_file)

for item in json_list[:10]:
    print(item["behavior_tree"])
    print("\n\n")

## Merge dataset

In [ ]:
# Merge both augmentation passes and remove duplicate BTs for the same original source.
with (DATASET_DIR / "synthetic_bt_dataset_first_run.json").open("r", encoding="utf-8") as file1:
    dataset1 = json.load(file1)

with (DATASET_DIR / "synthetic_bt_dataset_second_run.json").open("r", encoding="utf-8") as file2:
    dataset2 = json.load(file2)

unique_entries = set()
merged_data = []

for dataset_part in [dataset1, dataset2]:
    for entry in dataset_part:
        unique_key = (entry["original_bt_id"], entry["behavior_tree"])
        if unique_key not in unique_entries:
            unique_entries.add(unique_key)
            merged_data.append(entry)

for idx, entry in enumerate(merged_data):
    entry["id"] = idx

output_file = DATASET_DIR / "merged_bt_dataset.json"
with output_file.open("w", encoding="utf-8") as file:
    json.dump(merged_data, file, indent=4, ensure_ascii=False)

print(f"Merged dataset saved to '{output_file}'. Total entries: {len(merged_data)}")

### Convert TSE dataset in JSON format

In [ ]:
import glob
import os

# Convert the original TSE XML files into the same JSON schema used by generated BTs.
dir_path = "PATH_TO_TSE_DATASET"
filelist = sorted(glob.glob(os.path.join(dir_path, "**/*.xml"), recursive=True))

dataset = []
for idx, filepath in enumerate(filelist):
    try:
        with open(filepath, "r", encoding="utf-8") as file:
            behavior_tree = file.read()

        dataset.append({
            "id": idx,
            "original_bt_id": idx,
            "behavior_tree": behavior_tree
        })
    except Exception as e:
        print(f"Error processing file {filepath}: {e}")

output_file = DATASET_DIR / "TSE_dataset.json"
with output_file.open("w", encoding="utf-8") as json_file:
    json.dump(dataset, json_file, indent=4, ensure_ascii=False)

print(f"Dataset saved to '{output_file}'. Total entries: {len(dataset)}")

## Validate XML schema of BT

In [ ]:
from xml.etree import ElementTree as ET
from tqdm import tqdm

indexes = []
file_name = DATASET_DIR / "TSE_dataset.json"

with file_name.open("r", encoding="utf-8") as json_file:
    json_list = json.load(json_file)

# Record malformed XML entries so they can be removed before augmentation.
for i in tqdm(range(len(json_list))):
    try:
        ET.fromstring(json_list[i]["behavior_tree"])
    except ET.ParseError as e:
        print(f"Error parsing BT #{i}: {e}")
        indexes.append(i)

In [ ]:
# Remove invalid XML entries in reverse order so indices remain stable.
for index in sorted(indexes, reverse=True):
    del json_list[index]

In [ ]:
file_name = DATASET_DIR / "TSE_dataset_cleaned.json"
with file_name.open("w", encoding="utf-8") as json_file:
    json.dump(json_list, json_file, indent=4, ensure_ascii=False)